### Requirements

In [ ]:
!pip install pyautogui

In [3]:
# Base Modules
import math
import time
import numpy as np

# OpenCV
import cv2

# MediaPipe
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# PyAutoGUI
import pyautogui

## Parameters

In [10]:
# ==================== USER PARAMETERS ==================== #
# Camera Settings
camera_index = 700
resolution = [640, 480]
fps = 60

# Model Settings
model_path = '../models/hand_landmarker.task'
no_of_hands = 1
hand_confidence = 0.7
presence_confidence = 0.6
tracking_confidence = 0.6

# Gesture Settings
pinch_dist_max = 35
click_dist_max = 35

# Screen & Cursor Settings
frame_margin = 90        # Deadzone border inside camera frame so hand reaches corners easily
smoothing = 5            # Cursor smoothing factor (1 = raw/no smoothing, 5 = smooth, 10 = heavy smooth)
# ========================================================== #

# Get primary monitor resolution
screen_w, screen_h = pyautogui.size()
pyautogui.FAILSAFE = False  # Avoids crash if cursor bumps screen edges

# Tracking memory variables
prev_x, prev_y = screen_w // 2, screen_h // 2
is_clicked = False  # Debounce lock

### MediaPipe

In [11]:
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=no_of_hands,
    min_hand_detection_confidence=hand_confidence,
    min_hand_presence_confidence=presence_confidence,
    min_tracking_confidence=tracking_confidence
)
detector = vision.HandLandmarker.create_from_options(options)

## Gesture Logic

In [12]:
def process_air_mouse(frame, result, w, h):
    global prev_x, prev_y, is_clicked
    
    status_text = "Idle"
    
    # Draw the active bounding box where hand maps to screen
    cv2.rectangle(frame, (frame_margin, frame_margin), 
                  (w - frame_margin, h - frame_margin), (200, 200, 200), 1)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        # Extract landmarks: 4 = Thumb Tip, 8 = Index Tip, 12 = Middle Tip
        thumb_x, thumb_y = int(hand[4].x * w), int(hand[4].y * h)
        idx_x, idx_y = int(hand[8].x * w), int(hand[8].y * h)
        mid_x, mid_y = int(hand[12].x * w), int(hand[12].y * h)

        # Distances
        pinch_dist = math.hypot(thumb_x - idx_x, thumb_y - idx_y)
        click_dist = math.hypot(idx_x - mid_x, idx_y - mid_y)

        # Draw all 21 joints
        for lm in hand:
            cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 3, (0, 0, 255), cv2.FILLED)

        # ---------------- 1. MOVE CURSOR (Thumb + Index Pinch) ---------------- #
        if pinch_dist < pinch_dist_max:
            status_text = "Tracking Cursor"
            
            # Map index coordinates inside the margin to full monitor resolution
            target_x = np.interp(idx_x, (frame_margin, w - frame_margin), (0, screen_w))
            target_y = np.interp(idx_y, (frame_margin, h - frame_margin), (0, screen_h))

            # Apply smoothing (Exponential Moving Average)
            curr_x = prev_x + (target_x - prev_x) / smoothing
            curr_y = prev_y + (target_y - prev_y) / smoothing

            # Move physical mouse cursor
            pyautogui.moveTo(curr_x, curr_y)
            prev_x, prev_y = curr_x, curr_y

            # Visuals
            cv2.line(frame, (thumb_x, thumb_y), (idx_x, idx_y), (0, 255, 0), 2)
            cv2.circle(frame, (idx_x, idx_y), 7, (0, 255, 0), cv2.FILLED)
            cv2.circle(frame, (thumb_x, thumb_y), 7, (0, 255, 0), cv2.FILLED)

            # ---------------- 2. LEFT CLICK (Middle touches Index) ---------------- #
            if click_dist < click_dist_max:
                if not is_clicked:
                    pyautogui.click()
                    is_clicked = True  # Lock to fire only 1 click per touch
                status_text = "LEFT CLICK!"
                cv2.line(frame, (idx_x, idx_y), (mid_x, mid_y), (255, 255, 0), 2)
                cv2.circle(frame, (mid_x, mid_y), 9, (255, 255, 0), cv2.FILLED)
            else:
                is_clicked = False  # Reset lock when middle finger moves away
        else:
            is_clicked = False
            cv2.circle(frame, (idx_x, idx_y), 7, (0, 0, 255), cv2.FILLED)
            cv2.circle(frame, (thumb_x, thumb_y), 7, (255, 0, 255), cv2.FILLED)

    return frame, status_text

### Camera Configuration

In [13]:
cap = cv2.VideoCapture(camera_index)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, resolution[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, resolution[1])
cap.set(cv2.CAP_PROP_FPS, fps)

True

# Main Logic

In [14]:
print("Air Mouse Active! Move inside the white box. Press 'q' to stop.")

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape

    # MediaPipe inference
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    timestamp_ms = int(time.time() * 1000)
    result = detector.detect_for_video(mp_image, timestamp_ms)

    # Process hand movements & click triggers
    frame, status_text = process_air_mouse(frame, result, w, h)

    # HUD Status
    is_active = "Tracking" in status_text or "CLICK" in status_text
    cv2.putText(frame, status_text, (20, 40), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, 0.75, 
                (0, 255, 0) if is_active else (0, 0, 255), 2)

    cv2.imshow("Air Mouse", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Air Mouse Active! Move inside the white box. Press 'q' to stop.
